# Experimento C: Reward Shaping — Entorno Mejorado sobre 3 Capas de 128 (3.5M Pasos)

Implementación del **Experimento C** correspondiente al estudio comparativo de ingeniería de RL para Magic Jewelry.

| Experimento | Arquitectura | Pasos | ¿Qué cambió? | Récord anterior |
|:---|:---|:---|:---|:---|
| A | 2 × 128 | 2.000.000 | — Línea base | 12.060 pts |
| B | 3 × 128 | 3.500.000 | +1 capa oculta | 16.290 pts |
| **C** | **3 × 128** | **3.500.000** | **Entorno mejorado (Reward Shaping)** | ? |

### Hipótesis científica
El Experimento B demostró que la arquitectura de 3 capas supera el límite de la de 2 capas en un +35% de récord.
Sin embargo, **el cuello de botella ya no está en la arquitectura sino en la calidad de las señales de aprendizaje**.
El entorno actual tiene tres problemas específicos que este experimento corrige:

1. **Penalización de altura tardía:** En el entorno anterior, la penalización por altura solo se activaba cuando
   una columna superaba 8 filas de las 13 disponibles (`if altura_maxima > 8`). La IA no sentía ningún
   peligro durante las dos terceras partes de la partida.
   → **Corrección C:** Penalización continua y proporcional a la altura máxima desde el primer turno.

2. **Recompensa de cascadas no llega al agente:** El entorno anterior calculaba correctamente el multiplicador
   de cascadas en `_resolver_cascadas()` (`paso_combo`), pero ese multiplicador sólo afectaba a
   `puntos_turno`, que luego se escalaba a `× 0.1` antes de llegar al agente. El beneficio se diluía.
   → **Corrección C:** El multiplicador de cascadas afecta directamente a la recompensa del agente con
   un factor exponencial: cada ronda adicional de cascada vale 30% más que la anterior.

3. **Game over es una constante plana de -100:** Da lo mismo morir en el turno 5 que en el turno 130.
   La IA no aprende que conservar la vida en fases avanzadas es un objetivo autónomo.
   → **Corrección C:** Penalización escalonada = `-(15 + 0.5 × piezas_colocadas)`.

### Artefactos de salida:
1. `modelo_magic_jewelry_experimento_c.onnx`
2. `reporte_entrenamiento_experimento_c.csv`
3. `grafica_entrenamiento_experimento_c.png`

### 1. Instalación de Dependencias

In [ ]:
!pip install --quiet gymnasium stable-baselines3 onnx onnxruntime onnxscript pandas matplotlib

### 2. Entorno de Simulación v3 — Con Reward Shaping

Configuración del entorno v3 incorporando reward shaping continuo y multiplicador de cascadas.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np


class EntornoMagicJewelry(gym.Env):
    """
    Entorno v3 (Experimento C) — Incluye reward shaping sobre la misma
    lógica de juego de los experimentos A y B.

    Espacio de observación: 84 valores (idéntico a A y B).
    Espacio de acciones:    18 acciones discretas (idéntico a A y B).
    """

    metadata = {"render_modes": ["ansi"]}

    FILAS = 13
    COLUMNAS = 6
    NUM_COLORES_ESTANDAR = 6
    COLOR_GEMA_MAGICA = 7
    PUNTOS_POR_GEMA = 30
    JOYAS_POR_NIVEL = 70

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(18)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(84,), dtype=np.float32
        )
        self.tablero = np.zeros((self.FILAS, self.COLUMNAS), dtype=np.int8)
        self.pieza_actual    = np.zeros(3, dtype=np.int8)
        self.pieza_siguiente = np.zeros(3, dtype=np.int8)
        self.gemas_destruidas = 0
        self.siguiente_umbral_magica = self.JOYAS_POR_NIVEL
        self.proxima_es_magica = False
        self.puntaje_acumulado = 0
        self.piezas_colocadas  = 0

    # ──────────────────────────────────────────────────────────────
        # ──────────────────────────────────────────────────────────────

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.tablero.fill(0)
        self.puntaje_acumulado = 0
        self.piezas_colocadas  = 0
        self.gemas_destruidas  = 0
        self.siguiente_umbral_magica = self.JOYAS_POR_NIVEL
        self.proxima_es_magica = False
        self.pieza_actual    = self._generar_pieza_aleatoria()
        self.pieza_siguiente = self._generar_pieza_aleatoria()
        return self._obtener_observacion(), {}

    def step(self, action: int):
        columna  = int(action) // 3
        rotacion = int(action) % 3

        # 1. Preparar pieza (rotar si no es mágica)
        es_magica = (self.pieza_actual[0] == self.COLOR_GEMA_MAGICA)
        pieza_rotada = self._rotar_pieza(self.pieza_actual, rotacion) if not es_magica else self.pieza_actual.copy()

        tablero_previo = self.tablero.copy()

        # 2. Colocar la pieza en el tablero
        filas_ocupadas, desborde = self._colocar_y_caer(columna, pieza_rotada)
        self.piezas_colocadas += 1

        # 3. Bono de adyacencia de colores al colocar (igual que en A y B)
        bonus_adyacencia = 0.0
        if not es_magica and not desborde:
            conexiones = self._contar_conexiones_adyacentes(filas_ocupadas, columna, tablero_previo)
            bonus_adyacencia = conexiones * (self.PUNTOS_POR_GEMA * 0.25 * 0.1)

        # 4. Procesar impacto: gema mágica vs estándar
        if es_magica:
            gemas_eliminadas, puntos_turno = self._procesar_impacto_magica(filas_ocupadas, columna)
        else:
            # Factor de cascadas con escalado exponencial hacia el agente
            gemas_eliminadas, puntos_turno = self._resolver_cascadas_mejorado()

        # 5. Detectar fin de partida
        terminado = False
        if desborde and gemas_eliminadas == 0:
            terminado = True
        elif self.tablero[0, 3] != 0:
            terminado = True

        # 6. Función de recompensa mejorada
        recompensa = self._calcular_recompensa_v3(
            puntos_turno=puntos_turno,
            bonus_adyacencia=bonus_adyacencia,
            terminado=terminado
        )

        self.puntaje_acumulado += puntos_turno
        self.gemas_destruidas  += gemas_eliminadas

        # 7. Actualizar umbral de gema mágica
        if self.gemas_destruidas >= self.siguiente_umbral_magica:
            self.proxima_es_magica = True
            while self.siguiente_umbral_magica <= self.gemas_destruidas:
                self.siguiente_umbral_magica += self.JOYAS_POR_NIVEL

        # 8. Avanzar turno
        if not terminado:
            self.pieza_actual = self.pieza_siguiente.copy()
            if self.proxima_es_magica:
                self.pieza_siguiente = np.array([self.COLOR_GEMA_MAGICA] * 3, dtype=np.int8)
                self.proxima_es_magica = False
            else:
                self.pieza_siguiente = self._generar_pieza_aleatoria()

        info = {
            "puntaje": self.puntaje_acumulado,
            "piezas": self.piezas_colocadas,
            "gemas_destruidas": self.gemas_destruidas,
            "es_magica": es_magica
        }
        return self._obtener_observacion(), recompensa, terminado, False, info

    # ──────────────────────────────────────────────────────────────
    # Función de recompensa v3
    # ──────────────────────────────────────────────────────────────

    def _calcular_recompensa_v3(self, puntos_turno: int, bonus_adyacencia: float, terminado: bool) -> float:
        """
        Versión mejorada de la función de recompensa.

        Cambios respecto al Experimento B:
        - Penalización de altura: continua desde turno 1, proporcional a la altura
          máxima normalizada. Antes: solo se penalizaba si altura_max > 8.
        - Game over: penalización escalonada -(15 + 0.5 × piezas_colocadas).
          Antes: constante fija de -100.
        - El multiplicador de cascadas ya viene absorbido en puntos_turno desde
          _resolver_cascadas_mejorado(), por lo que el factor × 0.1 lo traslada.
        """
        if terminado:
            # Penalización temporal de fin de episodio
            return -(15.0 + 0.5 * self.piezas_colocadas)

        recompensa = (puntos_turno * 0.1) + bonus_adyacencia + 1.0

        # Penalización continua por altura desde el primer turno
        alturas = [
            self.FILAS - np.argmax(self.tablero[:, c] != 0)
            if np.any(self.tablero[:, c] != 0) else 0
            for c in range(self.COLUMNAS)
        ]
        altura_maxima = max(alturas)
        # Penalización proporcional: escala de 0.0 (tablero vacío) a -2.0 (columna llena)
        recompensa -= (altura_maxima / self.FILAS) * 2.0

        return float(recompensa)

    # ──────────────────────────────────────────────────────────────
    # Multiplicador exponencial por cascada
    # ──────────────────────────────────────────────────────────────

    def _resolver_cascadas_mejorado(self):
        """
        Resuelve las cascadas igual que antes, pero aplica un multiplicador de puntos
        que crece un 30% por cada ronda adicional de cascada.

        En el entorno anterior, paso_combo multiplicaba solo los puntos_turno,
        que luego se dividían entre 10 en la función de recompensa, diluyendo el incentivo.
        Aquí el multiplicador opera directamente sobre los puntos antes de escalarlos,
        por lo que el agente siente la diferencia entre destruir 6 gemas en 1 ronda
        vs. 3+3 en 2 rondas de cascada.
        """
        total_gemas   = 0
        puntos_totales = 0
        ronda = 0

        while True:
            matches = self._buscar_matches()
            if not matches:
                break

            ronda += 1
            cantidad = len(matches)
            total_gemas += cantidad

            # Multiplicador por nivel de cascada
            # Ronda 1: ×1.0 | Ronda 2: ×1.3 | Ronda 3: ×1.69 | ...
            multiplicador = 1.3 ** (ronda - 1)
            puntos_totales += int(cantidad * self.PUNTOS_POR_GEMA * multiplicador)

            for f, c in matches:
                self.tablero[f, c] = 0
            self._aplicar_gravedad()

        return total_gemas, puntos_totales

    # ──────────────────────────────────────────────────────────────
    # Métodos auxiliares (idénticos a Experimentos A y B)
    # ──────────────────────────────────────────────────────────────

    def _generar_pieza_aleatoria(self) -> np.ndarray:
        return self.np_random.integers(1, self.NUM_COLORES_ESTANDAR + 1, size=3, dtype=np.int8)

    def _rotar_pieza(self, pieza: np.ndarray, rotacion: int) -> np.ndarray:
        if rotacion == 0:
            return pieza.copy()
        elif rotacion == 1:
            return np.array([pieza[2], pieza[0], pieza[1]], dtype=np.int8)
        else:
            return np.array([pieza[1], pieza[2], pieza[0]], dtype=np.int8)

    def _colocar_y_caer(self, columna: int, pieza: np.ndarray):
        fila_primer_bloque = self.FILAS
        for f in range(self.FILAS):
            if self.tablero[f, columna] != 0:
                fila_primer_bloque = f
                break

        fila_asiento = fila_primer_bloque - 1
        desborde = False
        filas_ocupadas = []

        for i in range(3):
            fila_destino = fila_asiento - i
            gema = pieza[2 - i]
            if fila_destino >= 0:
                self.tablero[fila_destino, columna] = gema
                filas_ocupadas.append(fila_destino)
            else:
                desborde = True

        return filas_ocupadas, desborde

    def _contar_conexiones_adyacentes(self, filas_ocupadas: list, columna: int, tablero_previo: np.ndarray) -> int:
        conexiones = 0
        dirs = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
        for f in filas_ocupadas:
            color = self.tablero[f, columna]
            if color == 0:
                continue
            for df, dc in dirs:
                nf, nc = f + df, columna + dc
                if 0 <= nf < self.FILAS and 0 <= nc < self.COLUMNAS:
                    if tablero_previo[nf, nc] == color:
                        conexiones += 1
        return conexiones

    def _procesar_impacto_magica(self, filas_ocupadas: list, columna: int):
        if not filas_ocupadas:
            return 0, 0
        fila_asiento = max(filas_ocupadas)
        fila_debajo  = fila_asiento + 1
        gemas_eliminadas_color = 0
        if fila_debajo < self.FILAS:
            color_objetivo = self.tablero[fila_debajo, columna]
            if color_objetivo != 0 and color_objetivo != self.COLOR_GEMA_MAGICA:
                mascara = (self.tablero == color_objetivo)
                gemas_eliminadas_color = int(np.sum(mascara))
                self.tablero[mascara] = 0
        self.tablero[self.tablero == self.COLOR_GEMA_MAGICA] = 0
        self._aplicar_gravedad()
        total_destruidas = gemas_eliminadas_color + 3
        puntos_magica    = total_destruidas * self.PUNTOS_POR_GEMA
        gemas_cascadas, puntos_cascadas = self._resolver_cascadas_mejorado()
        return (total_destruidas + gemas_cascadas), (puntos_magica + puntos_cascadas)

    def _buscar_matches(self) -> set:
        gemas_a_eliminar = set()
        for f in range(self.FILAS):
            for c in range(self.COLUMNAS):
                color = self.tablero[f, c]
                if color == 0 or color == self.COLOR_GEMA_MAGICA:
                    continue
                if c + 2 < self.COLUMNAS and color == self.tablero[f, c+1] == self.tablero[f, c+2]:
                    gemas_a_eliminar.update({(f,c),(f,c+1),(f,c+2)})
                if f + 2 < self.FILAS and color == self.tablero[f+1, c] == self.tablero[f+2, c]:
                    gemas_a_eliminar.update({(f,c),(f+1,c),(f+2,c)})
                if f+2 < self.FILAS and c+2 < self.COLUMNAS and color == self.tablero[f+1,c+1] == self.tablero[f+2,c+2]:
                    gemas_a_eliminar.update({(f,c),(f+1,c+1),(f+2,c+2)})
                if f+2 < self.FILAS and c-2 >= 0 and color == self.tablero[f+1,c-1] == self.tablero[f+2,c-2]:
                    gemas_a_eliminar.update({(f,c),(f+1,c-1),(f+2,c-2)})
        return gemas_a_eliminar

    def _aplicar_gravedad(self):
        for c in range(self.COLUMNAS):
            col_datos  = self.tablero[:, c]
            gemas_vivas = col_datos[col_datos != 0]
            ceros = self.FILAS - len(gemas_vivas)
            self.tablero[:ceros, c] = 0
            self.tablero[ceros:, c] = gemas_vivas

    def _obtener_observacion(self) -> np.ndarray:
        tablero_norm   = (self.tablero.flatten() / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        actual_norm    = (self.pieza_actual    / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        siguiente_norm = (self.pieza_siguiente / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        return np.concatenate([tablero_norm, actual_norm, siguiente_norm])


print("Entorno v3 (Experimento C) definido correctamente.")
print(f"  Observaciones: {EntornoMagicJewelry().observation_space.shape[0]}")
print(f"  Acciones:      {EntornoMagicJewelry().action_space.n}")

### 3. Callback de Métricas por Episodio

In [ ]:
import os, time
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from stable_baselines3.common.callbacks import BaseCallback


class CallbackMetricasC(BaseCallback):
    """
    Registra puntaje, gemas_destruidas y piezas_colocadas de cada episodio
    y genera el CSV + gráfica al finalizar el entrenamiento.
    """

    def __init__(self, frecuencia_reporte: int = 100):
        super().__init__()
        self.frecuencia_reporte = frecuencia_reporte
        self.historial = []
        self.partida_num = 0
        self.tiempo_inicio = None
        self.directorio_salida = "/kaggle/working"

    def _on_training_start(self):
        self.tiempo_inicio = time.time()
        print("Experimento C iniciado. Entorno v3 con reward shaping activo.")

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if info.get("episode") or ("piezas" in info and "TimeLimit.truncated" not in info):
                if "piezas" in info:
                    self.partida_num += 1
                    self.historial.append({
                        "partida":           self.partida_num,
                        "paso_global":       self.num_timesteps,
                        "puntaje":           info.get("puntaje", 0),
                        "piezas_colocadas":  info.get("piezas",  0),
                        "gemas_destruidas":  info.get("gemas_destruidas", 0),
                    })
                    if self.partida_num % self.frecuencia_reporte == 0:
                        ultimas = self.historial[-self.frecuencia_reporte:]
                        df_tmp  = pd.DataFrame(ultimas)
                        mins_trans = (time.time() - self.tiempo_inicio) / 60
                        print(
                            f"[C] Partida {self.partida_num:>6} | "
                            f"Paso {self.num_timesteps:>9,} | "
                            f"Pts media: {df_tmp['puntaje'].mean():>7.1f} | "
                            f"Pts max: {df_tmp['puntaje'].max():>6} | "
                            f"Gemas max: {df_tmp['gemas_destruidas'].max():>4} | "
                            f"Tiempo: {mins_trans:.1f} min"
                        )
        return True

    def _on_training_end(self):
        mins_total = (time.time() - self.tiempo_inicio) / 60
        df = pd.DataFrame(self.historial)
        print(f"\n=== EXPERIMENTO C FINALIZADO en {mins_total:.1f} min ===")
        print(f"Total episodios: {len(df)}")
        print(f"Puntaje máximo: {df['puntaje'].max()} pts")
        print(f"Gemas destruidas máximas: {df['gemas_destruidas'].max()}")
        print(f"Media puntaje (últimos 10k eps): {df['puntaje'].tail(10000).mean():.1f}")

        ruta_csv = os.path.join(self.directorio_salida, "reporte_entrenamiento_experimento_c.csv")
        df.to_csv(ruta_csv, index=False)
        print(f"[C] CSV guardado en '{ruta_csv}'")

        fig, axes = plt.subplots(3, 1, figsize=(12, 9))
        fig.suptitle(
            "Experimento C: Reward Shaping — 3 Capas de 128 (3.5M Pasos)",
            fontsize=14, fontweight="bold"
        )
        axes[0].plot(df["puntaje"], alpha=0.2, color="#888888")
        axes[0].plot(df["puntaje"].rolling(50, min_periods=1).mean(),
                     color="#e63946", linewidth=2, label="Media móvil (50 eps)")
        axes[0].set_ylabel("Puntos")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(df["piezas_colocadas"], alpha=0.2, color="#888888")
        axes[1].plot(df["piezas_colocadas"].rolling(50, min_periods=1).mean(),
                     color="#2a9d8f", linewidth=2, label="Media móvil (50 eps)")
        axes[1].set_ylabel("Piezas colocadas")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(df["gemas_destruidas"], alpha=0.2, color="#888888")
        axes[2].plot(df["gemas_destruidas"].rolling(50, min_periods=1).mean(),
                     color="#8338ec", linewidth=2, label="Media móvil (50 eps)")
        axes[2].set_ylabel("Gemas eliminadas")
        axes[2].set_xlabel("Número de partida (Episodio)")
        axes[2].legend(); axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        ruta_png = os.path.join(self.directorio_salida, "grafica_entrenamiento_experimento_c.png")
        plt.savefig(ruta_png, dpi=200)
        plt.close()
        print(f"[C] Gráfica guardada en '{ruta_png}'")


print("Callback del Experimento C definido.")

### 4. Modelo PPO — Arquitectura 3 × 128 (idéntica al Experimento B)

La arquitectura **no cambia** respecto al Experimento B.
Esto garantiza que la única variable del experimento es el entorno.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

NUM_ENTORNOS = 8
env_vectorizado = make_vec_env(EntornoMagicJewelry, n_envs=NUM_ENTORNOS)

# Arquitectura: idéntica al Experimento B
policy_kwargs = dict(
    net_arch=dict(pi=[128, 128, 128], vf=[128, 128, 128])
)

modelo_c = PPO(
    policy="MlpPolicy",
    env=env_vectorizado,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=128,
    n_epochs=10,
    gamma=0.99,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs,
    verbose=0
)

print("Modelo PPO del Experimento C configurado.")
print("Arquitectura: 3 capas × 128 neuronas (Actor y Crítico)")
print("Hiperparámetros: idénticos al Experimento B")

### 5. Entrenamiento (3.500.000 Pasos / ~55 a 65 Minutos)

In [ ]:
callback_c = CallbackMetricasC(frecuencia_reporte=100)

PASOS_TOTALES = 3_500_000
print(f"--- INICIANDO EXPERIMENTO C ({PASOS_TOTALES:,} PASOS) ---\n")
print("Entorno:      v3 (reward shaping activo)")
print("Arquitectura: 3 × 128 (igual que Experimento B)")
print("Configuración de entrenamiento: 3.500.000 pasos.\n")

modelo_c.learn(total_timesteps=PASOS_TOTALES, callback=callback_c)
modelo_c.save("magic_jewelry_ppo_experimento_c_final")
print("\n--- ENTRENAMIENTO EXPERIMENTO C FINALIZADO ---")

### 6. Exportación a ONNX — Listo para el Navegador

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np


class PoliticaActorONNX(nn.Module):
    """Envuelve solo la red del Actor (pi) para exportar a ONNX."""
    def __init__(self, policy):
        super().__init__()
        self.mlp_extractor = policy.mlp_extractor.policy_net.to("cpu")
        self.action_net    = policy.action_net.to("cpu")

    def forward(self, x):
        caracteristicas = self.mlp_extractor(x)
        logits          = self.action_net(caracteristicas)
        return torch.argmax(logits, dim=-1)


# 1. Mover modelo entrenado a CPU
modelo_c.policy.to("cpu")
red_actor = PoliticaActorONNX(modelo_c.policy)
red_actor.eval()

# 2. Tensor de ejemplo en CPU
tensor_ejemplo = torch.zeros((1, 84), dtype=torch.float32, device="cpu")
NOMBRE_ONNX = "modelo_magic_jewelry_experimento_c.onnx"

# 3. Exportar
torch.onnx.export(
    red_actor,
    tensor_ejemplo,
    NOMBRE_ONNX,
    input_names=["observacion"],
    output_names=["accion"],
    opset_version=18
)

# 4. Verificar integridad
modelo_onnx = onnx.load(NOMBRE_ONNX)
onnx.checker.check_model(modelo_onnx)

# 5. Inferencia de prueba
sesion = ort.InferenceSession(NOMBRE_ONNX)
resultado = sesion.run(None, {"observacion": np.zeros((1, 84), dtype=np.float32)})

print(f"\n¡EXPERIMENTO C — ONNX GENERADO CON ÉXITO!: '{NOMBRE_ONNX}'")
print(f"Acción de prueba (tablero vacío): {resultado[0][0]}")
print("\nArtefactos de salida exportados:")
print("  1. modelo_magic_jewelry_experimento_c.onnx")
print("  2. reporte_entrenamiento_experimento_c.csv")
print("  3. grafica_entrenamiento_experimento_c.png")